In [25]:
import numpy as np
import pandas as pd

Question 1: Global Class Distribution & Clean Sample Mass
In data/public/train.csv, each of the 11,393 unique training images has 4 rows corresponding to the 4 defect classes (defect_a, defect_b, defect_c, defect_d), totaling 45,572 rows. A row with empty/null mask_rle indicates the absence of that defect.

Compute the frequency distribution of positive occurrences for each defect class, as well as the total number of completely clean images (images with no defect across all 4 classes).

Question: Based on your calculations, what is the exact sum of the count of positive samples in the most frequent defect class and the total count of completely clean images?

Answer format: Integer.

In [29]:
df = pd.read_csv(r'D:\SurfaceGuard_AI\data\train.csv')
df.head(10)

,image_id,defect_type,mask_rle,width,height
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128
1,4S22I7SA34,defect_b,NaN,800,128
2,4S22I7SA34,defect_c,NaN,800,128
3,4S22I7SA34,defect_d,NaN,800,128
4,DZ899111OE,defect_a,NaN,800,128
5,DZ899111OE,defect_b,NaN,800,128
6,DZ899111OE,defect_c,NaN,800,128
7,DZ899111OE,defect_d,NaN,800,128
8,GVDJ3PQ5QT,defect_a,NaN,800,128
9,GVDJ3PQ5QT,defect_b,NaN,800,128


In [34]:
defective_samples = df.loc[df['mask_rle'].notna()]
defective_samples.shape[0]

7201

In [37]:
most_frequent_count = defective_samples['defect_type'].value_counts().max()
most_frequent_count

np.int64(4100)

In [38]:
df['has_defect'] = df['mask_rle'].notnull()
df.head(5)

,image_id,defect_type,mask_rle,width,height,has_defect
0,4S22I7SA34,defect_a,7255 6 7377 12 7505 12 7633 12 7762 11 7890 11...,800,128,True
1,4S22I7SA34,defect_b,NaN,800,128,False
2,4S22I7SA34,defect_c,NaN,800,128,False
3,4S22I7SA34,defect_d,NaN,800,128,False
4,DZ899111OE,defect_a,NaN,800,128,False


In [41]:
defects_per_image = df.groupby('image_id')['has_defect'].sum()
clean_images_count = (defects_per_image==0).sum()
clean_images_count


np.int64(5516)

In [42]:
print(f"Most Frequent count + Clean Images count: {most_frequent_count+clean_images_count}")

Most Frequent count + Clean Images count: 9616


Question 2: Multi-Label Cardinality & Co-occurrence Complexity
While defect classes are reported across separate rows, industrial steel sheets can suffer from simultaneous manufacturing errors. Analyze the label cardinality (the number of distinct defect classes present per image) across all 11,393 images in train.csv.

Question: Exactly how many images in the training set exhibit multi-label defect co-occurrence (i.e., contain strictly more than 1 defect class)?

Answer format: Integer.

In [43]:
more_than_one_class_count = (defects_per_image>1).sum()
more_than_one_class_count

np.int64(1198)

Question 3: The Spatial Mutual-Exclusivity Proof (Architectural Insight)
Even if an image has multiple defects at the image level, the physical defects may or may not overlap spatially. Decode all masks for the multi-label images identified in Question 2 onto their native (128, 800) canvas.

Check whether any individual pixel carries annotations for two or more defect classes at the exact same spatial coordinate 
.

Question: What is the total count of overlapping defect pixels (pixels with 
 defect labels) across the entire training dataset?

Answer format: Short answer (exact count).

In [57]:
import os
import sys

sys.path.append(os.path.abspath('..'))
from src.utils import rle_to_mask
multi_label_image_ids = defects_per_image[defects_per_image>1].index
total_overlapping_pixels = 0

for img_id in multi_label_image_ids:
    img_df = df.loc[df['image_id']==img_id]
    img_df_defective = img_df[img_df['mask_rle'].notnull()]
    combined_mask = np.zeros((128,800))
    for rle_string,h,w in zip(img_df_defective['mask_rle'],img_df_defective['height'],img_df_defective['width']):
        decoded_mask  = rle_to_mask(rle_string,h,w)
        combined_mask+=decoded_mask
    overlap_count = np.sum(combined_mask>=2)
    total_overlapping_pixels+=overlap_count
print(total_overlapping_pixels)


0
